### RAG Pipelines- Data Ingestion to Vector DB Pipeline


In [1]:
import os

In [3]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf_files")


Found 2 PDF files to process

Processing: Github_cheat_sheet.pdf
  ✓ Loaded 2 pages

Processing: SQL_CheatSheet.pdf
  ✓ Loaded 8 pages

Total documents loaded: 10


In [7]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\Github_cheat_sheet.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Github_cheat_sheet.pdf', 'file_type': 'pdf'}, page_content='GIT CHEAT SHEET\nMAKE CHANGES\nReview edits and craft a commit transaction\n$ git status\nLists all new or modiﬁed ﬁles to be committed\n$ git add [file]\nSnapshots the ﬁle in preparation for versioning\n$ git reset [file]\nUnstages the ﬁle, but preserve its contents\n$ git diff\nShows ﬁle diﬀerences not yet staged\n$ git diff --staged\nShows ﬁle diﬀerences between staging and the last ﬁle version\n$ git commit -m "[descriptive message]"\nRecords ﬁle snapshots permanently in version history\nCONFIGURE TOOLING\nConﬁgure user information for all local repositories\n$ git config --global user.name "[name]"\nSets the name you want attached to your commit transactions\n$ git config --global user.email "[email address]"\nSets the emai

In [18]:
### Text splitting get into chunks

def split_pdf_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [19]:
chunks=split_pdf_documents(all_pdf_documents)
chunks

Split 10 documents into 59 chunks

Example chunk:
Content: GIT CHEAT SHEET
MAKE CHANGES
Review edits and craft a commit transaction
$ git status
Lists all new or modiﬁed ﬁles to be committed
$ git add [file]
Snapshots the ﬁle in preparation for versioning
$ g...
Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\Github_cheat_sheet.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Github_cheat_sheet.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\Github_cheat_sheet.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Github_cheat_sheet.pdf', 'file_type': 'pdf'}, page_content='GIT CHEAT SHEET\nMAKE CHANGES\nReview edits and craft a commit transaction\n$ git status\nLists all new or modiﬁed ﬁles to be committed\n$ git add [file]\nSnapshots the ﬁle in preparation for versioning\n$ git reset [file]\nUnstages the ﬁle, but preserve its contents\n$ git diff\nShows ﬁle diﬀerences not yet staged\n$ git diff --staged\nShows ﬁle diﬀerences between staging and the last ﬁle version\n$ git commit -m "[descriptive message]"\nRecords ﬁle snapshots permanently in version history\nCONFIGURE TOOLING\nConﬁgure user information for all local repositories\n$ git config --global user.name "[name]"\nSets the name you want attached to your commit transactions\n$ git config --global user.email "[email address]"\nSets the emai

### embedding And vectorStoreDB